# Lezione 12 — Multicollinearità, selezione delle variabili e validazione

**Corso:** Statistica Applicata con Python · **Blocco 4:** Modelli ed esperimenti
**Tempo stimato:** 105 minuti · **Capitolo del manuale:** `manual/12_multicolinealidad_seleccion_validacion_IT.md`

## Cosa vedrai in questa lezione

- **Multicollinearità** e **VIF**: perché due variabili molto simili gonfiano gli errori standard e rendono instabili i coefficienti.
- **Selezione delle variabili** con criterio: dominio, significatività, **AIC e BIC** (e l'R² corretto nei modelli lineari).
- Il pericolo della **selezione per ricerca**: scegliere tra 30 variabili di puro rumore.
- **Overfitting** e validazione **hold-out**, con NumPy e pandas.
- **Validazione incrociata k-fold** a mano e confronto con un singolo hold-out.
- **Classi sbilanciate**: accuratezza contro la previsione banale, **matrice di confusione**, precisione e recall.
- **Data leakage**: selezionare variabili con i dati di test.

## Prerequisiti

- Lezione 10 (OLS, R² corretto, AIC) e lezione 11 (modello logistico, log-odds, odds ratio).
- Librerie: `numpy`, `pandas`, `matplotlib`, `statsmodels` (`pip install numpy pandas matplotlib statsmodels`; sono già installate in Google Colab).

> **Come usare questo notebook:** esegui le celle in ordine. Prima di ogni esperimento scrivi cosa ti aspetti che accada. I dati di abbandono dei clienti (*churn*) sono sintetici: un *esempio illustrativo* generato con coefficienti veri noti, per poter confrontare la stima con la verità.

---
## Cella 0 — Setup

Importiamo le librerie, fissiamo il **seme** e definiamo il generatore di clienti della lezione 11 (con i suoi **coefficienti veri**, `VERDAD`), le metriche che calcoleremo a mano (`log_loss`, `exactitud`, `matriz_confusion`) e la validazione incrociata `kfold_log_loss`.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.outliers_influence import variance_inflation_factor
import warnings
warnings.filterwarnings("ignore")

rng = np.random.default_rng(42)          # seme fisso: tutti otteniamo gli stessi risultati
plt.rcParams["figure.figsize"] = (8, 4)
plt.rcParams["axes.grid"] = True

# Coefficienti VERI con cui generiamo i dati (li conosciamo solo perché sono sintetici)
VERDAD = {"Intercept": -1.8, "antiguedad": -0.03, "gasto": 0.03, "datos_gb": 0.0, "tickets": 0.45, "plan_anual": -0.9}

def generar_clientes(n, rng):
    """Genera n clienti sintetici (esempio illustrativo) con abbandono secondo VERDAD."""
    antiguedad = np.clip(rng.gamma(2.0, 14, n), 1, 72).round()
    gasto = np.clip(30 + 0.25 * antiguedad + rng.normal(0, 10, n), 10, None).round(1)
    datos_gb = np.clip(0.09 * gasto + rng.normal(0, 0.25, n), 0.1, None).round(2)   # molto correlata con gasto
    tickets = rng.poisson(np.exp(0.008 * (gasto - 40)))
    plan_anual = rng.binomial(1, 0.35, n)
    eta = (VERDAD["Intercept"] + VERDAD["antiguedad"] * antiguedad + VERDAD["gasto"] * gasto
           + VERDAD["tickets"] * tickets + VERDAD["plan_anual"] * plan_anual)
    abandona = rng.binomial(1, 1 / (1 + np.exp(-eta)))
    return pd.DataFrame({"abandona": abandona, "antiguedad": antiguedad, "gasto": gasto,
                         "datos_gb": datos_gb, "tickets": tickets, "plan_anual": plan_anual})

def log_loss(y, p):
    """Perdita logaritmica media (minore è meglio)."""
    y, p = np.asarray(y), np.asarray(p)
    return float(-np.mean(y * np.log(p) + (1 - y) * np.log(1 - p)))

def exactitud(y, p, umbral=0.5):
    """Proporzione di risposte corrette classificando con una soglia."""
    return float(np.mean((np.asarray(p) > umbral) == np.asarray(y)))

def matriz_confusion(y, p, umbral=0.5):
    """Restituisce (VP, FP, FN, VN) classificando 'abbandona' se p > soglia."""
    y, pred = np.asarray(y), np.asarray(p) > umbral
    return (int(((pred == 1) & (y == 1)).sum()), int(((pred == 1) & (y == 0)).sum()),
            int(((pred == 0) & (y == 1)).sum()), int(((pred == 0) & (y == 0)).sum()))

def kfold_log_loss(formula, d, k, rng):
    """Validazione incrociata k-fold a mano: restituisce il log-loss di ogni blocco."""
    orden = rng.permutation(len(d))
    folds = np.array_split(orden, k)
    perdidas = []
    for i in range(k):
        test = d.iloc[folds[i]]
        train = d.iloc[np.concatenate([folds[j] for j in range(k) if j != i])]
        m = smf.logit(formula, data=train).fit(disp=0)
        perdidas.append(log_loss(test["abandona"], m.predict(test)))
    return np.array(perdidas)

print("Librerie e funzioni caricate correttamente.")

---
## 1. I dati: abbandono dei clienti (*churn*)

Gli stessi 2 000 clienti della lezione 11. La risposta `abandona` vale 1 se il cliente disdice. Le esplicative: anzianità (mesi), spesa mensile (€), ticket di supporto, piano annuale (0/1) e GB di dati (`datos_gb`), quest'ultima **molto correlata con la spesa** e **senza effetto proprio** (coefficiente vero 0).

In [ ]:
df = generar_clientes(2000, rng)
idx = rng.permutation(len(df))     # ordine casuale delle righe per l'hold-out (sezione 7)

display(df.head())
print("Tasso di abbandono:", round(df["abandona"].mean() * 100, 1), "%")
print("Correlazione gasto - datos_gb:", round(df["gasto"].corr(df["datos_gb"]), 2))

---
## 2. Multicollinearità: cosa fa a un coefficiente

Adattiamo il modello logistico della lezione 11 **senza** e **con** `datos_gb` e confrontiamo il coefficiente e l'errore standard di `gasto`.

In [ ]:
FORM = "abandona ~ antiguedad + gasto + tickets + plan_anual"
m_sin = smf.logit(FORM, data=df).fit(disp=0)
m_con = smf.logit(FORM + " + datos_gb", data=df).fit(disp=0)

tabla = pd.DataFrame({
    "coef. di gasto": [m_sin.params["gasto"], m_con.params["gasto"]],
    "errore standard": [m_sin.bse["gasto"], m_con.bse["gasto"]],
    "p-value": [m_sin.pvalues["gasto"], m_con.pvalues["gasto"]]},
    index=["senza datos_gb", "con datos_gb"])
display(tabla.round(4))
print("Rapporto tra errori standard (con / senza):", round(m_con.bse["gasto"] / m_sin.bse["gasto"], 1))
print("Coefficiente vero di gasto:", VERDAD["gasto"])

**Cosa osservi?** Con `datos_gb` l'errore standard di `gasto` si moltiplica quasi per 4 e il suo p-value non è più significativo, anche se l'effetto è reale. Il coefficiente di `datos_gb`, il cui valore vero è 0, esce positivo e anch'esso con errore grande: il modello non sa ripartire l'effetto tra due variabili che si muovono quasi insieme.

### Esperimento guidato 1: l'instabilità dei coefficienti

Ripetiamo lo studio **200 volte** con campioni nuovi di 500 clienti, adattando il modello con e senza `datos_gb`, e guardiamo quanto varia il coefficiente di `gasto` da un campione all'altro. **Prima di eseguire, prevedi:** in quale modello varierà di più? E con quale frequenza uscirà con il segno sbagliato?

In [ ]:
rng_col = np.random.default_rng(123)
coef_sin, coef_con = [], []
for _ in range(200):
    d = generar_clientes(500, rng_col)
    coef_sin.append(smf.logit(FORM, data=d).fit(disp=0).params["gasto"])
    coef_con.append(smf.logit(FORM + " + datos_gb", data=d).fit(disp=0).params["gasto"])

print("Coefficiente vero di gasto:", VERDAD["gasto"])
print("Senza datos_gb → media:", round(np.mean(coef_sin), 4), " dev. standard:", round(np.std(coef_sin, ddof=1), 4))
print("Con datos_gb → media:", round(np.mean(coef_con), 4), " dev. standard:", round(np.std(coef_con, ddof=1), 4))
print("% di campioni con segno sbagliato (con datos_gb):", round(np.mean(np.array(coef_con) < 0) * 100, 1))

plt.hist(coef_sin, bins=25, alpha=0.6, label="senza datos_gb")
plt.hist(coef_con, bins=25, alpha=0.6, label="con datos_gb")
plt.axvline(VERDAD["gasto"], color="black", linestyle="--", label="verità")
plt.title("Coefficiente di gasto in 200 campioni")
plt.legend()
plt.show()

**Cosa osservi?** Le due medie sono vicine alla verità (0,03), ma con `datos_gb` la dispersione è circa quattro volte maggiore e in più di un campione su cinque il coefficiente esce **negativo**. Il modello è corretto in media e poco affidabile nel singolo campione. Questo significa "coefficienti instabili".

---
## 3. Rilevare la multicollinearità: il VIF

`VIF = 1 / (1 − R²)`, dove R² è quello di spiegare quella variabile con tutte le altre. Lo calcoliamo in tre modi: a mano con la correlazione, con la regressione ausiliaria e con `variance_inflation_factor`.

In [ ]:
columnas = ["antiguedad", "gasto", "datos_gb", "tickets", "plan_anual"]

r = df["gasto"].corr(df["datos_gb"])
print("VIF a mano con sole due variabili, 1/(1-r²) con r = 0,97:", round(1 / (1 - 0.97 ** 2), 1))

aux = smf.ols("gasto ~ antiguedad + datos_gb + tickets + plan_anual", data=df).fit()
print("R² della regressione ausiliaria di gasto:", round(aux.rsquared, 3), "→ VIF =", round(1 / (1 - aux.rsquared), 2))

def tabla_vif(datos, cols):
    Xm = sm.add_constant(datos[cols])
    return pd.Series({c: variance_inflation_factor(Xm.values, i) for i, c in enumerate(Xm.columns) if c != "const"})

display(pd.DataFrame({"con datos_gb": tabla_vif(df, columnas)}).round(2))
display(pd.DataFrame({"senza datos_gb": tabla_vif(df, ["antiguedad", "gasto", "tickets", "plan_anual"])}).round(2))

**Cosa osservi?** I due VIF alti (≈ 17) sono proprio la coppia `gasto` – `datos_gb`; gli altri sono vicini a 1. Eliminando `datos_gb` tutti tornano a valori ≈ 1. Ricorda: le soglie (5, 10) sono convenzioni; la decisione su quale variabile eliminare è di **dominio** (qui, la spesa è ciò che l'azienda fattura e controlla).

---
## 4. Scegliere le variabili: R² corretto nei modelli lineari

Nella lezione 10 abbiamo visto l'R² corretto. Verifichiamo che l'R² normale **sale sempre** aggiungendo variabili inutili e che R² corretto, AIC e BIC lo rilevano. Modello lineare con 300 clienti e risposta `gasto`; aggiungiamo 3 colonne di puro rumore.

In [ ]:
rl = np.random.default_rng(31)
dl = generar_clientes(300, rl)
for j in range(1, 4):
    dl[f"nz{j}"] = rl.normal(size=300)

cand_lin = {"solo antiguedad": "gasto ~ antiguedad",
            "+ 3 variabili di rumore": "gasto ~ antiguedad + nz1 + nz2 + nz3"}
filas = []
for nombre, f in cand_lin.items():
    m = smf.ols(f, data=dl).fit()
    filas.append([m.rsquared, m.rsquared_adj, m.aic, m.bic])
display(pd.DataFrame(filas, index=list(cand_lin), columns=["R²", "R² corretto", "AIC", "BIC"]).round(4))

**Cosa osservi?** L'R² **sale** con le tre variabili di rumore (anche se non spiegano nulla); l'R² corretto **scende** e AIC e BIC **peggiorano** (salgono). Una qualsiasi delle tre misure corrette rileva che le variabili sono di troppo; l'R² normale no.

---
## 5. AIC e BIC nel modello logistico

`AIC = 2k − 2·LL` e `BIC = k·ln(n) − 2·LL`, con `k` parametri (contando la costante) e `LL` la log-verosimiglianza. Verifichiamo la formula a mano con il modello M3 e confrontiamo i cinque candidati.

In [ ]:
m3 = smf.logit(FORM, data=df).fit(disp=0)
k, n, LL = len(m3.params), int(m3.nobs), m3.llf
print("k =", k, "  n =", n, "  LL =", round(LL, 1))
print("AIC a mano:", round(2 * k - 2 * LL, 1), "  statsmodels:", round(m3.aic, 1))
print("BIC a mano:", round(k * np.log(n) - 2 * LL, 1), "  statsmodels:", round(m3.bic, 1))

candidatos = {
    "M0: antiguedad": "abandona ~ antiguedad",
    "M1: + tickets": "abandona ~ antiguedad + tickets",
    "M2: + plan_anual": "abandona ~ antiguedad + tickets + plan_anual",
    "M3: + gasto": "abandona ~ antiguedad + tickets + plan_anual + gasto",
    "M4: + datos_gb": "abandona ~ antiguedad + tickets + plan_anual + gasto + datos_gb"}
ajustes = {nombre: smf.logit(f, data=df).fit(disp=0) for nombre, f in candidatos.items()}
tabla_ic = pd.DataFrame({"k": [len(m.params) for m in ajustes.values()],
                         "AIC": [m.aic for m in ajustes.values()],
                         "BIC": [m.bic for m in ajustes.values()]}, index=list(ajustes))
display(tabla_ic.round(1))
print("AIC minore:", tabla_ic["AIC"].idxmin(), "  |  BIC minore:", tabla_ic["BIC"].idxmin())

**Cosa osservi?** Il calcolo a mano coincide con `statsmodels`. I due criteri scendono fino a M3 e salgono in M4: `datos_gb` non compensa il suo parametro extra. Con 2 000 clienti AIC e BIC coincidono; il BIC penalizza ogni parametro con ln(2000) ≈ 7,6 contro 2 dell'AIC. Coincidono sempre? Lo vediamo ora.

### Esperimento guidato 2: AIC contro BIC con campioni di 500 clienti

Prima un campione concreto e poi 200 campioni nuovi di 500 clienti: quale modello sceglie ciascun criterio? Il modello vero è M3. **Prima di eseguire, prevedi:** chi sceglierà più spesso un modello troppo grande? E troppo piccolo?

In [ ]:
d500 = generar_clientes(500, np.random.default_rng(100))
aj500 = {nombre: smf.logit(f, data=d500).fit(disp=0) for nombre, f in candidatos.items() if nombre[:2] in ("M2", "M3", "M4")}
display(pd.DataFrame({"AIC": [m.aic for m in aj500.values()], "BIC": [m.bic for m in aj500.values()]}, index=list(aj500)).round(1))

elige_aic, elige_bic = np.zeros(5, int), np.zeros(5, int)
r_sim = np.random.default_rng(2025)
for _ in range(200):
    d = generar_clientes(500, r_sim)
    aj = [smf.logit(f, data=d).fit(disp=0) for f in candidatos.values()]
    elige_aic[np.argmin([m.aic for m in aj])] += 1
    elige_bic[np.argmin([m.bic for m in aj])] += 1
res_ic = pd.DataFrame({"AIC (% di campioni)": elige_aic / 2, "BIC (% di campioni)": elige_bic / 2},
                      index=["M0", "M1", "M2", "M3", "M4"]).T
display(res_ic)

**Cosa osservi?** Nel campione concreto gli AIC di M2 e M3 quasi coincidono (differenza < 2: indistinguibili) e il BIC preferisce il più semplice. Nei 200 campioni, l'AIC indovina il modello vero (M3) più spesso ma a volte tiene la variabile ridondante (M4); il BIC non sceglie quasi mai M4 ma con 500 clienti si ferma spesso a M2 e **omette `gasto`**, il cui effetto è reale anche se modesto. Non c'è un vincitore universale: AIC orientato alla previsione, BIC alla parsimonia.

---
## 6. Il pericolo della selezione per ricerca

### Esperimento guidato 3: selezione sul puro rumore

200 ripetizioni. In ciascuna: 200 clienti, una risposta 0/1 **generata a caso** e **30 variabili di rumore**. Adattiamo il modello con tutte e 30, teniamo quelle con p < 0,05, riadattiamo e guardiamo il p-value migliore. **Prima di eseguire, prevedi:** se tutto è rumore, quante ripetizioni finiranno con un modello "convincente"?

In [ ]:
rng_sw = np.random.default_rng(11)
n_elegidas, p_min_final = [], []
for _ in range(200):
    Xr = pd.DataFrame(rng_sw.normal(size=(200, 30)), columns=[f"x{j}" for j in range(30)])
    yr = rng_sw.binomial(1, 0.3, 200)
    completo = sm.Logit(yr, sm.add_constant(Xr)).fit(disp=0, maxiter=200)
    elegidas = [c for c in Xr.columns if completo.pvalues[c] < 0.05]
    n_elegidas.append(len(elegidas))
    if elegidas:
        final = sm.Logit(yr, sm.add_constant(Xr[elegidas])).fit(disp=0)
        p_min_final.append(final.pvalues[elegidas].min())

n_elegidas, p_min_final = np.array(n_elegidas), np.array(p_min_final)
print("Variabili scelte per ripetizione (media):", round(n_elegidas.mean(), 2), "(per caso se ne attendono 1,5)")
print("% di ripetizioni con almeno una variabile scelta:", round((n_elegidas > 0).mean() * 100, 1))
print("Di queste, % con almeno una p < 0,05 nel modello finale:", round((p_min_final < 0.05).mean() * 100, 1))
print("Mediana del p-value minimo del modello finale:", round(np.median(p_min_final), 3))

**Cosa osservi?** Nella maggior parte delle ripetizioni si "scoprono" variabili utili che **non esistono**, con p-value finali piccoli. I p-value del modello riadattato sono ottimisti perché le variabili sono state scelte proprio perché sembravano buone. Per questo: candidate definite con criterio di dominio prima di guardare i dati e valutazione con dati non usati per scegliere.

---
## 7. Overfitting e validazione hold-out

Separiamo 1 400 clienti di addestramento e 600 di test (con l'ordine casuale `idx` della sezione 1), stimiamo sull'addestramento e valutiamo **a mano** con log-loss e accuratezza. Come riferimento usiamo la previsione costante (il tasso di abbandono di addestramento).

In [ ]:
train = df.iloc[idx[:1400]]
test = df.iloc[idx[1400:]]
m_train = smf.logit(FORM, data=train).fit(disp=0)

p_tr, p_te = m_train.predict(train), m_train.predict(test)
p_base = np.full(len(test), train["abandona"].mean())      # previsione costante

resultados = pd.DataFrame({
    "log-loss": [log_loss(train["abandona"], p_tr), log_loss(test["abandona"], p_te), log_loss(test["abandona"], p_base)],
    "accuratezza": [exactitud(train["abandona"], p_tr), exactitud(test["abandona"], p_te), exactitud(test["abandona"], p_base)]},
    index=["modello, addestramento", "modello, test", "costante, test"])
display(resultados.round(3))
print("Accuratezza di 'nessuno abbandona' nel test:", round((test["abandona"] == 0).mean(), 3))

**Cosa osservi?** Addestramento e test danno quasi lo stesso risultato: con 4 variabili e 1 400 clienti non c'è overfitting. Nota l'**accuratezza**: il modello supera appena "nessuno abbandona" (circa 0,77 contro 0,765), anche se il suo log-loss è nettamente migliore di quello della previsione costante. Con classi sbilanciate l'accuratezza inganna (sezione 9).

### Esperimento guidato 4: aggiungere variabili di rumore

30 ripetizioni con **200 clienti di addestramento** e 1 000 di test, aggiungendo `k` = 0, 5, 10, 20 e 30 variabili di puro rumore alle 4 reali. **Prima di eseguire, prevedi:** cosa succede al log-loss in addestramento e in test all'aumentare di `k`?

In [ ]:
def simular_sobreajuste(n_train, k, repeticiones=30, semilla=7):
    r = np.random.default_rng(semilla)
    perdidas = []
    for _ in range(repeticiones):
        d = generar_clientes(n_train + 1000, r)
        for j in range(k):
            d[f"nz{j + 1}"] = r.normal(size=len(d))
        tr, te = d.iloc[:n_train], d.iloc[n_train:]
        formula = FORM + "".join(f" + nz{j + 1}" for j in range(k))
        m = smf.logit(formula, data=tr).fit(disp=0, maxiter=200)
        perdidas.append((log_loss(tr["abandona"], m.predict(tr)), log_loss(te["abandona"], m.predict(te))))
    return np.mean(perdidas, axis=0)

ks = [0, 5, 10, 20, 30]
res_sobre = pd.DataFrame([simular_sobreajuste(200, k) for k in ks], index=ks,
                         columns=["log-loss addestramento", "log-loss test"])
res_sobre.index.name = "variabili di rumore"
display(res_sobre.round(3))

res_sobre.plot(marker="o")
plt.axhline(log_loss(test["abandona"], p_base), color="gray", linestyle="--", label="previsione costante")
plt.title("L'adattamento in addestramento migliora, il test peggiora")
plt.ylabel("log-loss")
plt.legend()
plt.show()

**Cosa osservi?** Ogni variabile inutile fa **scendere** il log-loss di addestramento (sembra che il modello si adatti meglio) e **salire** quello di test. Con molte variabili di rumore il modello rende peggio con dati nuovi della previsione costante (≈ 0,55). Il divario tra le due curve è l'overfitting, e si vede solo se valutiamo con dati che il modello non ha visto.

---
## 8. Validazione incrociata k-fold, a mano

Con 300 clienti dividiamo i dati in 5 blocchi da 60. Ogni blocco fa da test una volta e il modello si stima con gli altri quattro. Usiamo la funzione `kfold_log_loss` della Cella 0 (leggila: sono solo dieci righe di NumPy e pandas).

In [ ]:
d300 = generar_clientes(300, np.random.default_rng(5))
perd = kfold_log_loss(FORM, d300, 5, np.random.default_rng(1))
print("Log-loss per blocco:", perd.round(3))
print("Media (stima 5-fold):", round(perd.mean(), 3))

### Esperimento guidato 5: k-fold contro un singolo hold-out

Con gli stessi 300 clienti ripetiamo **200 volte** ciascuna procedura con partizioni casuali diverse: (a) un hold-out 70/30 e (b) un 5-fold (media dei 5 blocchi). **Prima di eseguire, prevedi:** avranno la stessa media? Quale varierà di più?

In [ ]:
ho, cv = [], []
for s in range(200):
    rr = np.random.default_rng(1000 + s)
    o = rr.permutation(300)
    a, b = d300.iloc[o[:210]], d300.iloc[o[210:]]
    m = smf.logit(FORM, data=a).fit(disp=0)
    ho.append(log_loss(b["abandona"], m.predict(b)))
    cv.append(kfold_log_loss(FORM, d300, 5, rr).mean())
ho, cv = np.array(ho), np.array(cv)

comparacion = pd.DataFrame({"media": [ho.mean(), cv.mean()], "dev. standard": [ho.std(ddof=1), cv.std(ddof=1)],
                            "minimo": [ho.min(), cv.min()], "massimo": [ho.max(), cv.max()]},
                           index=["hold-out 70/30", "5-fold"])
display(comparacion.round(3))
print("Rapporto tra deviazioni standard (hold-out / 5-fold):", round(ho.std(ddof=1) / cv.std(ddof=1), 1))

plt.hist(ho, bins=25, alpha=0.6, label="hold-out 70/30")
plt.hist(cv, bins=25, alpha=0.6, label="5-fold")
plt.title("Log-loss stimato in 200 partizioni diverse")
plt.legend()
plt.show()

**Cosa osservi?** Le medie sono quasi uguali, ma l'hold-out cambia enormemente con la partizione (intervallo di circa 0,35 punti di log-loss): con la partizione che ti capita potresti concludere che il modello è eccellente o mediocre. Il 5-fold, che testa con tutti i dati, è circa otto volte più stabile, al prezzo di adattare il modello cinque volte.

---
## 9. Classi sbilanciate: matrice di confusione, precisione e recall

Con il modello `m_train` e i 600 clienti di test della sezione 7 classifichiamo "abbandona" se `p > soglia`, contiamo i quattro risultati possibili e calcoliamo accuratezza, precisione e recall **a mano**.

In [ ]:
p_test, y_test = m_train.predict(test).values, test["abandona"].values
vp, fp, fn, vn = matriz_confusion(y_test, p_test, 0.5)

display(pd.DataFrame([[vp, fp], [fn, vn]],
                     index=["previsto: abbandona", "previsto: non abbandona"],
                     columns=["reale: abbandona", "reale: non abbandona"]))
print("Accuratezza = (VP + VN) / totale =", round((vp + vn) / len(y_test), 3))
print("Precisione = VP / (VP + FP) =", round(vp / (vp + fp), 3))
print("Recall = VP / (VP + FN) =", round(vp / (vp + fn), 3))
print("'Nessuno abbandona': accuratezza", round(np.mean(y_test == 0), 3), "  recall = 0")

**Cosa osservi?** L'accuratezza (0,77) sembra buona, ma "nessuno abbandona" azzecca già il 76,5 % con recall 0. Il nostro modello, con la soglia di default, individua solo una minoranza di quelli che se ne vanno. E se spostiamo la soglia?

In [ ]:
filas = []
for umbral in (0.5, 0.3, 0.2):
    vp, fp, fn, vn = matriz_confusion(y_test, p_test, umbral)
    filas.append([umbral, vp, fp, fn, vn, (vp + vn) / len(y_test), vp / (vp + fp), vp / (vp + fn)])
display(pd.DataFrame(filas, columns=["soglia", "VP", "FP", "FN", "VN", "accuratezza", "precisione", "recall"]).round(3))

**Cosa osservi?** Abbassando la soglia **sale il recall** (individuiamo più abbandoni) e **scende la precisione** (più falsi allarmi); l'accuratezza addirittura peggiora. Non c'è una soglia "giusta" in astratto: dipende dal costo di perdere un cliente rispetto a quello di un'offerta di fidelizzazione non necessaria. (Curve ROC/AUC: corso di Machine Learning.)

---
## 10. Data leakage: selezionare variabili con i dati di test

### Esperimento guidato 6: la fuga di informazione

200 ripetizioni. In ciascuna: 100 righe, 40 variabili di **puro rumore** e una risposta 0/1 a caso (50 %): nessun modello onesto può azzeccare più di ≈ 50 %. Scegliamo le 5 variabili più correlate con la risposta e misuriamo l'accuratezza sul 30 % di test in due modi: **(a) con fuga**, scegliendo con le 100 righe e separando dopo; **(b) corretto**, scegliendo solo con il 70 % di addestramento. **Prima di eseguire, prevedi** l'accuratezza media di ciascuna procedura.

In [ ]:
def elegir_top(Xm, y, filas, k=5):
    corr = np.abs([np.corrcoef(Xm[filas, j], y[filas])[0, 1] for j in range(Xm.shape[1])])
    return np.argsort(corr)[-k:]

rl2 = np.random.default_rng(3)
acc_fuga, acc_ok = [], []
for _ in range(200):
    Xm = rl2.normal(size=(100, 40))
    y = rl2.binomial(1, 0.5, 100)
    orden = rl2.permutation(100)
    tr_i, te_i = orden[:70], orden[70:]
    for con_fuga, lista in ((True, acc_fuga), (False, acc_ok)):
        cols = elegir_top(Xm, y, np.arange(100)) if con_fuga else elegir_top(Xm, y, tr_i)
        m = sm.Logit(y[tr_i], sm.add_constant(Xm[tr_i][:, cols])).fit(disp=0)
        pr = m.predict(sm.add_constant(Xm[te_i][:, cols], has_constant="add"))
        lista.append(((pr > 0.5) == y[te_i]).mean() * 100)

print("Accuratezza media sul test, CON fuga (%):", round(np.mean(acc_fuga), 1))
print("Accuratezza media sul test, procedura corretta (%):", round(np.mean(acc_ok), 1))
print("Errore standard della media (fuga):", round(np.std(acc_fuga, ddof=1) / np.sqrt(200), 1))

**Cosa osservi?** Con la fuga, un modello costruito sul **puro rumore** sembra azzeccare più del 60 %; con la procedura corretta ottiene ≈ 50 %, che corrisponde al caso. La fuga non dà alcun avviso: solo risultati troppo buoni. Regola: ogni passo che guarda la risposta si fa **solo con l'addestramento** (e dentro ogni blocco nella validazione incrociata).

---
## Riepilogo della lezione

- La **multicollinearità** gonfia gli errori standard e rende instabili i coefficienti senza danneggiare le previsioni; si rileva con il **VIF** e si risolve eliminando o combinando variabili, con criterio di dominio.
- Si scelgono le variabili con dominio, significatività e **AIC/BIC** (o R² corretto nei modelli lineari); il BIC penalizza di più la complessità e i due possono discordare.
- Cercare tra molte variabili dà p-value ottimisti: definisci le candidate prima di guardare e **valuta con dati non usati**.
- L'**overfitting** si vede confrontando addestramento e test; un solo hold-out è instabile, il **k-fold** è più stabile.
- Con classi sbilanciate l'accuratezza inganna: usa la **matrice di confusione**, la precisione e il recall.
- Il **data leakage** (scegliere variabili con i dati di test) fa sembrare buono un modello inutile.

**Prossimo passo:** il notebook di pratica di questa lezione e poi la lezione 13: Test A/B e disegno degli esperimenti.